Notebook para preparação geral dos dados

In [1]:
from google.colab import files
import io

uploaded = files.upload()

file_name = list(uploaded.keys())[0]

Saving compas-scores-two-years (1).csv to compas-scores-two-years (1) (2).csv


In [2]:
import pandas as pd

df_raw = pd.read_csv(file_name)
print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (7214, 53)


,id,name,first,last,compas_screening_date,sex,dob,age,age_cat,race,...,v_decile_score,v_score_text,v_screening_date,in_custody,out_custody,priors_count.1,start,end,event,two_year_recid
0,1,miguel hernandez,miguel,hernandez,2013-08-14,Male,1947-04-18,69,Greater than 45,Other,...,1,Low,2013-08-14,2014-07-07,2014-07-14,0,0,327,0,0
1,3,kevon dixon,kevon,dixon,2013-01-27,Male,1982-01-22,34,25 - 45,African-American,...,1,Low,2013-01-27,2013-01-26,2013-02-05,0,9,159,1,1
2,4,ed philo,ed,philo,2013-04-14,Male,1991-05-14,24,Less than 25,African-American,...,3,Low,2013-04-14,2013-06-16,2013-06-16,4,0,63,0,1
3,5,marcu brown,marcu,brown,2013-01-13,Male,1993-01-21,23,Less than 25,African-American,...,6,Medium,2013-01-13,NaN,NaN,1,0,1174,0,0
4,6,bouthy pierrelouis,bouthy,pierrelouis,2013-03-26,Male,1973-01-22,43,25 - 45,Other,...,1,Low,2013-03-26,NaN,NaN,2,0,1102,0,0


In [3]:
# ── Distribuição racial ─────────────────────────────────
print("=== DISTRIBUIÇÃO RACIAL ===")
print(df_raw['race'].value_counts())
print(f"\nPorcentagem:")
print(df_raw['race'].value_counts(normalize=True).mul(100).round(2))

=== DISTRIBUIÇÃO RACIAL ===
race
African-American    3696
Caucasian           2454
Hispanic             637
Other                377
Asian                 32
Native American       18
Name: count, dtype: int64

Porcentagem:
race
African-American    51.23
Caucasian           34.02
Hispanic             8.83
Other                5.23
Asian                0.44
Native American      0.25
Name: proportion, dtype: float64


In [3]:
# ── Filtros metodológicos (ProPublica) ──────────────────

df = df_raw.copy()
print(f"Shape inicial: {df.shape}")

# Filtro 1 — Reincidência indeterminada
df = df[df['is_recid'] != -1]
print(f"Após remover is_recid == -1: {df.shape}")

# Filtro 2 — Remover crimes de trânsito
df = df[df['c_charge_degree'] != 'O']
print(f"Após remover crimes de trânsito: {df.shape}")

# Filtro 3 — Score COMPAS deve ser válido
df = df[df['score_text'] != 'N/A']
print(f"Após remover score inválido: {df.shape}")

# Filtro 4 — Diferença entre crime e avaliação COMPAS <= 30 dias
df['compas_screening_date'] = pd.to_datetime(df['compas_screening_date'])
df['c_offense_date'] = pd.to_datetime(df['c_offense_date'])
df = df[((df['compas_screening_date'] - df['c_offense_date']).dt.days >= -30) &
        ((df['compas_screening_date'] - df['c_offense_date']).dt.days <=  30)]
print(f"Após filtro de 30 dias: {df.shape}")

# Filtro 5 — Manter apenas raças com representação suficiente
valid_races = ['African-American', 'Caucasian']
df = df[df['race'].isin(valid_races)]
print(f"Após filtro de raças: {df.shape}")

print(f"\nDistribuição racial após filtros:")
print(df['race'].value_counts())

Shape inicial: (7214, 53)
Após remover is_recid == -1: (7214, 53)
Após remover crimes de trânsito: (7214, 53)
Após remover score inválido: (7214, 53)
Após filtro de 30 dias: (5425, 53)
Após filtro de raças: (4610, 53)

Distribuição racial após filtros:
race
African-American    2732
Caucasian           1878
Name: count, dtype: int64


In [4]:
# ── Remoção de colunas desnecessárias ───────────────────

COLS_TO_DROP = [
    # Identificação pessoal
    'id', 'name', 'first', 'last', 'dob', 'age',
    # Duplicatas
    'decile_score.1', 'priors_count.1',
    # Datas usadas nos filtros
    'compas_screening_date', 'c_offense_date',
    # Metadados operacionais
    'screening_date', 'c_jail_in', 'c_jail_out',
    'c_arrest_date', 'in_custody', 'out_custody',
    'days_b_screening_arrest', 'c_case_number',
    'c_days_from_compas', 'c_charge_desc', 'r_case_number',
    'r_days_from_arrest', 'r_offense_date', 'r_charge_desc',
    'r_jail_in', 'r_jail_out', 'r_charge_degree',
    'decile_score', 'score_text', 'type_of_assessment',
    # Reincidência violenta — fora do escopo
    'violent_recid', 'is_violent_recid',
    'vr_case_number', 'vr_charge_degree', 'vr_charge_desc', 'vr_offense_date',
    # Score COMPAS de violência — fora do escopo
    'v_decile_score', 'v_score_text', 'v_type_of_assessment', 'v_screening_date',
    # Dados de sobrevivência — fora do escopo
    'start', 'end', 'event',
    # Target alternativo
    'is_recid'
]

df = df.drop(columns=COLS_TO_DROP)

print(f"Shape após remoção: {df.shape}")
print(f"\nColunas restantes:\n{df.columns.tolist()}")

Shape após remoção: (4610, 9)

Colunas restantes:
['sex', 'age_cat', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree', 'two_year_recid']


In [5]:
# ── Encoding ────────────────────────────────────────────

df_prepared = df.copy()

# Binários
df_prepared['sex_encoded'] = (df_prepared['sex'] == 'Male').astype(int)
df_prepared['c_charge_degree_encoded'] = (df_prepared['c_charge_degree'] == 'F').astype(int)
df_prepared['race_encoded'] = (df_prepared['race'] == 'African-American').astype(int)

# Ordinal
age_order = {'Less than 25': 0, '25 - 45': 1, 'Greater than 45': 2}
df_prepared['age_cat_encoded'] = df_prepared['age_cat'].map(age_order)

print(f"Shape: {df_prepared.shape}")
print(f"\nColunas:\n{df_prepared.columns.tolist()}")
print(f"\nAmostra:")
df_prepared.head()

Shape: (4610, 13)

Colunas:
['sex', 'age_cat', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree', 'two_year_recid', 'sex_encoded', 'c_charge_degree_encoded', 'race_encoded', 'age_cat_encoded']

Amostra:


,sex,age_cat,race,juv_fel_count,juv_misd_count,juv_other_count,priors_count,c_charge_degree,two_year_recid,sex_encoded,c_charge_degree_encoded,race_encoded,age_cat_encoded
1,Male,25 - 45,African-American,0,0,0,0,F,1,1,1,1,1
2,Male,Less than 25,African-American,0,0,1,4,F,1,1,1,1,0
3,Male,Less than 25,African-American,0,1,0,1,F,0,1,1,1,0
6,Male,25 - 45,Caucasian,0,0,0,14,F,1,1,1,0,1
8,Female,25 - 45,Caucasian,0,0,0,0,M,0,0,0,0,1


In [6]:
# ── Exportar dataset preparado ─────────────────────────

df_prepared.to_csv('/content/compas_prepared.csv', index=False)

print(f"compas_prepared.csv exportado com sucesso")
print(f"Shape: {df_prepared.shape}")
print(f"Colunas: {df_prepared.columns.tolist()}")

compas_prepared.csv exportado com sucesso
   Shape: (4610, 13)
   Colunas: ['sex', 'age_cat', 'race', 'juv_fel_count', 'juv_misd_count', 'juv_other_count', 'priors_count', 'c_charge_degree', 'two_year_recid', 'sex_encoded', 'c_charge_degree_encoded', 'race_encoded', 'age_cat_encoded']
